# Dexwin Pay — live assessment notebook

Read `README.md` and `data/DATA_CARD.md` first.

Goal: **catch fraud, cut false declines.** The cells below are a weak baseline. Improve them, then be ready to talk serving and monitoring.

Ignore `INTERVIEWER.md`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DATA_PATH = next(
    p for p in (Path("data/transactions.csv"), Path("../data/transactions.csv"))
    if p.exists()
)
df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

In [ ]:
print(df_raw.dtypes)
print()
print(df_raw["is_fraud"].value_counts(normalize=True))

## Weak baseline

This matches `scripts/baseline.py`. It is a starting point, not a target design.

In [ ]:
df = df_raw.copy()
n_loaded = len(df)

df["amount"] = (
    df["amount"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
    .replace("", pd.NA)
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

df = df.dropna().copy()

y = df["is_fraud"]
X = df.drop(columns=["is_fraud"])

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=250)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"rows loaded:       {n_loaded}")
print(f"rows after dropna: {len(df)}")
print(f"accuracy:          {accuracy_score(y_test, pred):.4f}")
print(confusion_matrix(y_test, pred))
# TODO: product asked for fewer false declines — add that later.

## Your work

Clean the table, choose features that would exist at authorization time, pick a metric that matches the product, and train something you would consider shadowing.

Do not spend the whole session on hyperparameter search.

In [ ]:
# Your cleaning / features / model.


In [ ]:
# Metrics you would actually report to the product owner.


## Serving and monitoring (write bullets, then talk)

Freeze the model around here and answer in words:

1. What calls the model, and what is in the request?
2. Which features are **not** safe to serve online from this warehouse dump?
3. How do you roll it out without instantly changing every merchant’s decline rate?
4. Fraud labels arrive late. What do you monitor **tomorrow morning** vs next week?
5. How do you notice leakage / feedback loops once the model is live?

In [ ]:
# Optional notes for the discussion (not graded as code).
